# Extract Entities

List all entities (people, places, objects, brands, concepts) found across your videos and images.
Use this for building a content index, understanding who and what appears in your videos and images, or feeding entity lists to downstream classification systems.

# Install the TwelveLabs Python SDK

In [ ]:
%pip install twelvelabs

In [ ]:
import json
import os

from twelvelabs import TwelveLabs, TextParam
from twelvelabs.types.text_param_format import TextParamFormat_JsonSchema

# Configuration
API_KEY = os.environ.get("TWELVELABS_API_KEY", "<YOUR_API_KEY>")
STORE_ID = os.environ.get("TWELVELABS_STORE_ID", "<YOUR_KNOWLEDGE_STORE_ID>")  # Replace with your knowledge store ID

client = TwelveLabs(api_key=API_KEY)

## Helper Functions

A utility to extract text content from a Jockey API response.

In [ ]:
def parse_response(response) -> str:
    """Extract text content from a Jockey response.

    Args:
        response: The ResponseObject returned by client.responses.create().

    Returns:
        The text content from the first message output, or an empty string
        if no message content is found.
    """
    for output in response.output:
        if output.type == "message":
            for content in output.content:
                return content.text
    return ""

## Entity Schema

Define a JSON schema for structured entity extraction. Each entity includes its name, type
(person, place, object, brand, or concept), frequency of appearance, and which videos and images it appears in.

In [ ]:
ENTITY_SCHEMA = {
    "type": "object",
    "properties": {
        "entities": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "type": {"type": "string"},
                    "frequency": {"type": "string"},
                    "appears_in": {
                        "type": "array",
                        "items": {"type": "string"},
                    },
                },
            },
        },
        "entity_count": {"type": "integer"},
    },
}

## Extract All Entities

Ask Jockey to scan every video and image in the knowledge store and return a comprehensive entity list.
The prompt requests all distinct entities with their frequency information.

In [ ]:
response = client.responses.create(
    knowledge_store_id=STORE_ID,
    input=[
        {
            "type": "message",
            "role": "user",
            "content": (
                "List every distinct entity across all videos and images -- people, places, "
                "objects, brands, and concepts. Include how frequently each appears."
            ),
        }
    ],
    text=TextParam(
        format=TextParamFormat_JsonSchema(name="entity_list", schema_=ENTITY_SCHEMA)
    ),
)

data = json.loads(parse_response(response))

print(f"Found {data['entity_count']} entities:\n")
for entity in data["entities"]:
    print(f"  [{entity['type']}] {entity['name']} -- {entity['frequency']}")
    if entity.get("appears_in"):
        print(f"    Videos: {', '.join(entity['appears_in'])}")

## Variations

Modify the prompt to tailor entity extraction to your needs:

- **Filter by type:** "List only the people who appear in these videos and images"
- **Cross-video tracking:** "Which entities appear in more than one video?"
- **With relationships:** "List entities and how they relate to each other"

## Next Steps

- **[Get Corpus Overview](get_corpus_overview.ipynb)** -- understand the full collection context
- **[Search Videos](search_videos.ipynb)** -- find specific moments by description
- **[Find Organization Axes](find_organization_axes.ipynb)** -- discover the best categorization strategies
- **[Enrich Content](enrich_content.ipynb)** -- get deeper, domain-specific metadata

See also:
- [Structured Output Guide](https://docs.twelvelabs.io/v1.3/agents/guides/create-a-response/structured-output) -- more on JSON schema responses